# H1 + H2: MedSigLIP controls (frozen features, same subsets as the pipeline)
**H1 - does a medical backbone beat the ImageNet ViT?** Compare linear probes (and MLP heads) on frozen ImageNet ViT-B/16 CLS features vs frozen **MedSigLIP-448** image embeddings, at every training fraction.

**H2 - is the "text gain" really MedSigLIP's image features in disguise?** MedGemma looks at images through a MedSigLIP-style vision encoder, so its captions may just re-express what that encoder sees. Here the text streams are swapped for the MedSigLIP image embedding inside the same frozen fusion head:

| Variant | Streams | Question it answers |
|---|---|---|
| `lp_imagenet` / `lp_medsiglip` | one backbone, linear head | H1 |
| `mlp_imagenet` / `mlp_medsiglip` | one backbone, fusion MLP | H1 (same head as the fusion runs) |
| `imagenet+text` | ImageNet CLS + description + answers | the frozen text result without the local branch |
| `imagenet+medsiglip` | ImageNet CLS + MedSigLIP embedding | **H2**: does the image embedding give the same gain as text? |
| `imagenet+medsiglip_shuffled` | MedSigLIP embedding of *another* image | control: the gain must be image-specific |
| `imagenet+medsiglip+text` | both | does text add anything *beyond* MedSigLIP? |
| `medsiglip+text` | MedSigLIP + text | same, on the medical backbone |
| `text_only` | description + answers | reference |

**Reading the outcome**
- `imagenet+medsiglip` >= `imagenet+text`, and `imagenet+medsiglip+text` ~ `imagenet+medsiglip`: the captions were an expensive proxy for the image encoder; drop the 11 s/image captioning.
- `imagenet+medsiglip+text` clearly > `imagenet+medsiglip`: language adds something beyond the encoder; H6 (text as a training-time teacher) is worth building.
- `lp_medsiglip` >> `lp_imagenet`: H1 holds; DKA on MedSigLIP is the next experiment.

Training is identical to the pipeline notebooks (Adam, 100 epochs, LR 1e-4 for the final layer and 1e-3 for projections, batch 8 / 32, modality dropout), so `mlp_imagenet` and `imagenet+text` should land close to the pipeline's `global_mlp` / `global+desc`/`global+qa` numbers. Linear probes use a parameter-free per-sample LayerNorm on their input so that the two backbones' different feature scales do not decide the comparison; the pipeline's raw-feature `linear_probe` is shown alongside for reference.

**Needs:** `google/medsiglip-448` is gated on Hugging Face (Health AI Developer Foundations terms, like MedGemma): accept them with the account that owns `HF_TOKEN`. Run the pipeline notebooks first (this notebook reuses their `split.csv`, `vit_feats.pt` and caption files). MedSigLIP embeddings are extracted once and cached. COVID 100% is off by default (it needs embeddings for all 16,932 training images); add `1.0` to its `fractions` in Cell 0 for the full-data check.

In [ ]:
# CELL 0 - Setup and config
import os, re, json, time, glob, gc, random, hashlib, math, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

def _load_env():                                   # HF_TOKEN from the nearest .env
    d = os.path.abspath(os.getcwd())
    while True:
        f = os.path.join(d, ".env")
        if os.path.isfile(f):
            for line in open(f, encoding="utf8"):
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    k, v = line.split("=", 1); os.environ.setdefault(k.strip(), v.strip().strip("'\""))
            return f
        if os.path.dirname(d) == d: return None
        d = os.path.dirname(d)
print(".env:", _load_env())

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = "./mm_medsiglip"; os.makedirs(OUT_DIR, exist_ok=True)
RESULTS_CSV = f"{OUT_DIR}/results.csv"; PREDS_CSV = f"{OUT_DIR}/preds.csv"

MEDSIGLIP_ID = "google/medsiglip-448"
EMB_BATCH = 16                                     # images per MedSigLIP forward pass (lower if you run out of GPU memory)
EMB_FP16 = True                                    # fp16 autocast for extraction (bf16 is slow on RDNA2); False = fp32

# training: identical to the pipeline notebooks
EPOCHS, BATCH, BIG_BATCH = 100, 8, 32
LR_HEAD, LR_MODULES = 1e-4, 1e-3
HID, P_DROP, P_MOD = 128, 0.3, 0.2

ALL_FR = [0.0063, 0.0125, 0.125, 1.0]
DATASETS = {
    "BUSI": dict(pipeline_dir="./mm_pipeline_busi", seed=42, pool_frac=0.25, fractions=ALL_FR,
                 hard_leak=r"benign|malignan|cancer|carcinom|bi-?rads|neoplas"),
    # COVID 100% needs MedSigLIP embeddings for all 16,932 training images (the slowest step, roughly 1-2 h on an
    # RX 6700 XT; an estimate). Add 1.0 here when you want the full-data check against the DKA paper.
    "COVID_full": dict(pipeline_dir="./mm_pipeline_covid_full", seed=42, pool_frac=0.05, fractions=[0.0063, 0.0125, 0.125],
                       hard_leak=r"covid|sars|coronavirus|pneumoni"),
}
SEEDS_FOR = {0.0063: [0, 1, 2, 3, 4], 0.0125: [0, 1, 2, 3, 4], 0.125: [0, 1, 2], 1.0: [0, 1]}
RUN_ORDER = ["BUSI", "COVID_full"]

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")


In [ ]:
# CELL 1 - Splits and training subsets, rebuilt exactly as in the pipeline notebooks
def prepare(name):
    cfg = DATASETS[name]
    df = pd.read_csv(f"{cfg['pipeline_dir']}/split.csv")
    df["id"] = df["image_path"].apply(lambda p: hashlib.md5(p.encode()).hexdigest()[:10] + "_" + os.path.basename(p))
    classes = sorted(df["label"].unique()); y = df["label"].map({c: i for i, c in enumerate(classes)}).values
    tr_all = np.where(df["split"].values == "train")[0]; te = np.where(df["split"].values == "test")[0]
    r = np.random.RandomState(cfg["seed"]); pool = []
    for c in range(len(classes)):
        full = tr_all[y[tr_all] == c]
        pool.append(r.permutation(full)[:max(1, int(round(cfg["pool_frac"] * len(full))))])
    pool = np.concatenate(pool)
    def subset(frac, seed):
        rng = np.random.RandomState(seed); out = []
        for c in range(len(classes)):
            full = tr_all[y[tr_all] == c]; n_c = max(1, int(round(frac * len(full))))
            src = pool[y[pool] == c] if frac <= cfg["pool_frac"] else full
            src = src[rng.permutation(len(src))]; out.append(src[:min(n_c, len(src))])
        return np.concatenate(out)
    pv2 = f"{cfg['pipeline_dir']}/results_v2.csv"
    pr = pd.read_csv(pv2 if os.path.exists(pv2) else f"{cfg['pipeline_dir']}/results_fast.csv")
    for f in cfg["fractions"]:
        for s in SEEDS_FOR[f]:
            n = pr[(pr["frac"].round(4) == round(f, 4)) & (pr["seed"] == s)]["n_train"]
            assert n.empty or int(n.iloc[0]) == len(subset(f, s)), f"{name} {f} seed {s}: subset differs from the pipeline"
    return dict(df=df, y=y, classes=classes, test=te, subset=subset, cfg=cfg, n=len(df))

D = {name: prepare(name) for name in RUN_ORDER}
for name, d in D.items():
    print(f"{name}: {d['n']} images | test {len(d['test'])} | "
          + ", ".join(f"{f*100:g}% -> {len(d['subset'](f, 0))}" for f in d["cfg"]["fractions"]))


In [ ]:
# CELL 2 - Frozen features: ImageNet CLS (from the pipeline cache) and MedSigLIP image embeddings (extracted once, cached)
from transformers import AutoModel, AutoImageProcessor   # image side only: no tokenizer / sentencepiece needed

def needed_rows(name):
    d = D[name]; s = set(d["test"].tolist())
    for f in d["cfg"]["fractions"]:
        for seed in SEEDS_FOR[f]: s |= set(d["subset"](f, seed).tolist())
    return np.array(sorted(s))

def imagenet_cls(name):
    d = D[name]; blob = torch.load(f"{d['cfg']['pipeline_dir']}/vit_feats.pt", mmap=True)   # mmap: don't pull ~2 GB of patch tokens into RAM
    if "ids" in blob: assert blob["ids"] == d["df"]["id"].tolist(), "pipeline features are for a different split"
    assert blob["cls"].shape[0] == d["n"]
    return blob["cls"].float().clone()

def medsiglip_emb(name):
    d = D[name]; cache = f"{OUT_DIR}/{name}_medsiglip448.pt"; rows = needed_rows(name)
    ids = d["df"]["id"].values[rows].tolist()
    if os.path.exists(cache):
        c = torch.load(cache)
        if set(ids) <= set(c["ids"]):
            pos = {i: k for k, i in enumerate(c["ids"])}; E = torch.zeros(d["n"], c["emb"].shape[1])
            for r, i in zip(rows, ids): E[r] = c["emb"][pos[i]].float()
            print(name, "MedSigLIP embeddings from cache"); return E, rows
    tok = os.environ.get("HF_TOKEN")
    model = AutoModel.from_pretrained(MEDSIGLIP_ID, token=tok).to(DEVICE).eval()
    proc = AutoImageProcessor.from_pretrained(MEDSIGLIP_ID, token=tok)
    out = []
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=EMB_FP16 and DEVICE == "cuda"):
        for i in tqdm(range(0, len(rows), EMB_BATCH), desc=f"{name} MedSigLIP"):
            ims = [Image.open(d["df"]["image_path"][r]).convert("RGB") for r in rows[i:i + EMB_BATCH]]
            px = proc(images=ims, return_tensors="pt")["pixel_values"].to(DEVICE)
            f = model.get_image_features(pixel_values=px)
            f = f if torch.is_tensor(f) else f.pooler_output     # transformers 5 returns an output object; pooler_output = the 1152-d embedding
            out.append(f.float().cpu())
    emb = torch.cat(out)
    assert torch.isfinite(emb).all(), "non-finite MedSigLIP features: set EMB_FP16 = False and re-run"
    torch.save({"ids": ids, "emb": emb.half(), "model": MEDSIGLIP_ID}, cache)
    del model; gc.collect(); torch.cuda.empty_cache()
    E = torch.zeros(d["n"], emb.shape[1]); E[rows] = emb
    print(name, f"MedSigLIP: {len(rows)} images, dim {emb.shape[1]}"); return E, rows

FEAT = {}
for name in RUN_ORDER:
    E, rows = medsiglip_emb(name)
    has = torch.zeros(D[name]["n"], dtype=torch.bool); has[torch.as_tensor(rows)] = True
    FEAT[name] = dict(g=imagenet_cls(name), m=E, has_m=has)
    print(name, "| ImageNet CLS", tuple(FEAT[name]["g"].shape), "| MedSigLIP", tuple(E.shape))


In [ ]:
# CELL 3 - Text features: BiomedCLIP embeddings of the pipeline's current captions (cached per caption file)
import open_clip
def load_text(name):
    d = D[name]; cfg = d["cfg"]
    info = json.load(open(f"{cfg['pipeline_dir']}/run_info.json"))
    caps = os.path.join(cfg["pipeline_dir"], os.path.basename(info["caps_file"]))
    tag = os.path.basename(caps); cache = f"{OUT_DIR}/{name}_text_{tag.replace('.jsonl', '')}.pt"
    if os.path.exists(cache):
        print(name, "text embeddings from cache", tag); return torch.load(cache)
    rows = {r["id"]: r for r in map(json.loads, open(caps)) if r.get("parse_ok")}
    leak = re.compile(cfg["hard_leak"], re.I)
    redact = lambda s: re.sub(r"\s+", " ", leak.sub("", s)).strip()
    ids = [i for i in d["df"]["id"] if i in rows]
    bm = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
    model, _ = open_clip.create_model_from_pretrained(bm); tk = open_clip.get_tokenizer(bm); model = model.to(DEVICE).eval()
    @torch.no_grad()
    def emb(texts, bs=64):
        return torch.cat([F.normalize(model.encode_text(tk(texts[i:i + bs], context_length=256).to(DEVICE)).float(), dim=-1).cpu()
                          for i in range(0, len(texts), bs)])
    nq = len(rows[ids[0]]["answers"])
    de = emb([redact(rows[i]["description"]) for i in ids])
    qa = emb([redact(a) for i in ids for a in rows[i]["answers"]]).reshape(len(ids), nq, -1).mean(1)
    pos = {i: k for k, i in enumerate(d["df"]["id"])}
    Dm, Qm, has = torch.zeros(d["n"], 512), torch.zeros(d["n"], 512), torch.zeros(d["n"], dtype=torch.bool)
    for k, i in enumerate(ids):
        Dm[pos[i]] = de[k]; Qm[pos[i]] = F.normalize(qa[k], dim=0); has[pos[i]] = True
    t = dict(D=Dm, Q=Qm, has=has, tag=tag, questions=info.get("questions"))
    torch.save(t, cache); del model; gc.collect(); torch.cuda.empty_cache()
    print(name, f"embedded {len(ids)} captions from {tag}"); return t

TEXT = {name: load_text(name) for name in RUN_ORDER}
for name, t in TEXT.items():
    print(name, "| captions:", t["tag"], "| first question:", (t["questions"] or ["?"])[0][:80])


In [ ]:
# CELL 4 - Models and training engine (same recipe as the pipeline; saves per-image predictions)
from sklearn.metrics import accuracy_score, f1_score, recall_score
DIMS = {"g": 768, "m": None, "d": 512, "q": 512}          # m filled in from the MedSigLIP embedding size

class LinearProbe(nn.Module):
    def __init__(s, key, dim, n_cls): super().__init__(); s.key = key; s.fc = nn.Linear(dim, n_cls)
    def forward(s, b): x = b[s.key]; return s.fc(F.layer_norm(x, x.shape[-1:]))   # parameter-free LN: scale-fair across backbones

class Fusion(nn.Module):
    def __init__(s, keys, n_cls):
        super().__init__(); s.keys = list(keys)
        s.enc = nn.ModuleDict({k: nn.Sequential(nn.LayerNorm(DIMS[k]), nn.Linear(DIMS[k], HID)) for k in s.keys})
        s.head = nn.Sequential(nn.Dropout(P_DROP), nn.Linear(len(s.keys) * HID, HID), nn.GELU(), nn.Linear(HID, n_cls))
    def forward(s, b):
        z = torch.stack([s.enc[k](b[k]) for k in s.keys], 1)
        if s.training and z.size(1) > 1 and P_MOD > 0:
            m = (torch.rand(z.size(0), z.size(1), device=z.device) > P_MOD).float()
            none = m.sum(1) == 0
            if none.any(): m[none.nonzero().squeeze(1), torch.randint(0, z.size(1), (int(none.sum()),), device=z.device)] = 1
            z = z * m.unsqueeze(-1)
        return s.head(z.flatten(1))

VARIANTS = {
    "lp_imagenet":                 dict(linear="g"),
    "lp_medsiglip":                dict(linear="m"),
    "mlp_imagenet":                dict(keys="g"),
    "mlp_medsiglip":               dict(keys="m"),
    "imagenet+text":               dict(keys="gdq"),
    "imagenet+medsiglip":          dict(keys="gm"),
    "imagenet+medsiglip_shuffled": dict(keys="gm", shuffle="m"),
    "imagenet+medsiglip+text":     dict(keys="gmdq"),
    "medsiglip+text":              dict(keys="mdq"),
    "text_only":                   dict(keys="dq"),
}
uses = lambda spec: set(spec.get("keys", "")) | ({spec["linear"]} if "linear" in spec else set())
needs_text = lambda spec: bool(uses(spec) & {"d", "q"})
COLS = ["dataset", "variant", "frac", "seed", "n_train", "acc", "f1", "sen", "text_tag"]

def derange(a, rng):
    if len(a) < 2: return a
    while True:
        p = a[rng.permutation(len(a))]
        if (p != a).all(): return p

def train_eval(name, spec, tr, seed):
    d, f_, t = D[name], FEAT[name], TEXT[name]; te = d["test"]; set_seed(seed)
    DIMS["m"] = f_["m"].shape[1]
    src = {"g": f_["g"], "m": f_["m"], "d": t["D"], "q": t["Q"]}
    src = {k: v.to(DEVICE) for k, v in src.items()}
    y = torch.as_tensor(d["y"], device=DEVICE).long()
    model = (LinearProbe(spec["linear"], DIMS[spec["linear"]], len(d["classes"])) if "linear" in spec
             else Fusion(spec["keys"], len(d["classes"]))).to(DEVICE)
    if "linear" in spec: groups = [{"params": model.parameters(), "lr": LR_HEAD}]
    else:
        fin = {id(p) for p in model.head[-1].parameters()}
        groups = [{"params": list(model.head[-1].parameters()), "lr": LR_HEAD},
                  {"params": [p for p in model.parameters() if id(p) not in fin], "lr": LR_MODULES}]
    opt = torch.optim.Adam(groups)
    m_tr, m_te = tr.copy(), te.copy()                       # which image's MedSigLIP embedding each image gets
    if spec.get("shuffle") == "m":
        rng = np.random.RandomState(seed + 999); m_tr, m_te = derange(m_tr, rng), derange(m_te, rng)
    def batch(rows, mrows):
        r, mr = torch.as_tensor(rows, device=DEVICE), torch.as_tensor(mrows, device=DEVICE)
        return {"g": src["g"][r], "m": src["m"][mr], "d": src["d"][r], "q": src["q"][r]}
    n = len(tr); bs = BIG_BATCH if n >= 128 else BATCH
    for ep in range(EPOCHS):
        model.train(); perm = np.random.permutation(n)
        for i in range(0, n, bs):
            b = perm[i:i + bs]
            loss = F.cross_entropy(model(batch(tr[b], m_tr[b])), y[torch.as_tensor(tr[b], device=DEVICE)])
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval(); logits = []
    with torch.no_grad():
        for i in range(0, len(te), 512): logits.append(model(batch(te[i:i + 512], m_te[i:i + 512])).float().cpu())
    prob = torch.cat(logits).softmax(1).numpy(); p = prob.argmax(1); yt = d["y"][te]
    return dict(acc=100 * accuracy_score(yt, p), f1=f1_score(yt, p, average="macro"), sen=100 * recall_score(yt, p, average="macro")), prob

def load_results():
    r = pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else pd.DataFrame(columns=COLS)
    r["text_tag"] = r["text_tag"].fillna(""); return r

def save_preds(name, v, f, s, tag, prob):
    d = D[name]; te = d["test"]
    out = pd.DataFrame({"dataset": name, "variant": v, "frac": round(f, 4), "seed": s, "text_tag": tag,
                        "image_id": d["df"]["id"].values[te], "y_true": d["y"][te], "y_pred": prob.argmax(1)})
    for k in range(prob.shape[1]): out[f"p{k}"] = prob[:, k].round(5)
    out.to_csv(PREDS_CSV, mode="a", header=not os.path.exists(PREDS_CSV), index=False)

def run(name, variants=None):
    d, t, f_ = D[name], TEXT[name], FEAT[name]
    res = load_results(); done = set(zip(res["dataset"], res["variant"], res["frac"].round(4), res["seed"], res["text_tag"]))
    jobs = []
    for f in d["cfg"]["fractions"]:
        for s in SEEDS_FOR[f]:
            tr = d["subset"](f, s)
            for v in (variants or VARIANTS):
                spec = VARIANTS[v]; tag = t["tag"] if needs_text(spec) else ""
                if (name, v, round(f, 4), s, tag) in done: continue
                if needs_text(spec) and not (t["has"][tr].all() and t["has"][d["test"]].all()): continue   # no captions here
                if "m" in uses(spec): assert f_["has_m"][tr].all() and f_["has_m"][d["test"]].all()
                jobs.append((f, s, v, tr, tag))
    print(f"{name}: {len(jobs)} runs to do")
    for f, s, v, tr, tag in tqdm(jobs, desc=name):
        m, prob = train_eval(name, VARIANTS[v], tr, s)
        save_preds(name, v, f, s, tag, prob)
        row = dict(dataset=name, variant=v, frac=round(f, 4), seed=s, n_train=len(tr), **m, text_tag=tag)
        pd.DataFrame([row])[COLS].to_csv(RESULTS_CSV, mode="a", header=not os.path.exists(RESULTS_CSV), index=False)


In [ ]:
# CELL 5 - BUSI runs
run("BUSI")


In [ ]:
# CELL 6 - COVID (full) runs
run("COVID_full")


In [ ]:
# CELL 7 - Results: H1 and H2 tables, paired t-tests over seeds, and McNemar tests pooled over seeds
from scipy import stats
from scipy.stats import binomtest
def read_preds(path):
    """BUSI (3 classes) and COVID (4) share one file, so the header lists p0-p2 while COVID rows carry p3 too."""
    head = open(path, encoding="utf8").readline().strip().split(",")
    base = [c for c in head if not (c[:1] == "p" and c[1:].isdigit())]
    return pd.read_csv(path, names=base + [f"p{k}" for k in range(16)], header=None, skiprows=1).dropna(axis=1, how="all")
res = load_results(); preds = read_preds(PREDS_CSV)
order = list(VARIANTS)

def mcnemar_pooled(name, f, a, b):
    """Exact McNemar test on the discordant pairs pooled over seeds (same test images, same subsets)."""
    tag = TEXT[name]["tag"]
    p = preds[(preds["dataset"] == name) & (preds["frac"].round(4) == round(f, 4)) & preds["text_tag"].fillna("").isin(["", tag])]
    A = p[p["variant"] == a].set_index(["seed", "image_id"]); B = p[p["variant"] == b].set_index(["seed", "image_id"])
    j = A.join(B, lsuffix="_a", rsuffix="_b", how="inner")
    if j.empty: return None
    ca, cb = j["y_pred_a"] == j["y_true_a"], j["y_pred_b"] == j["y_true_b"]
    n10, n01 = int((ca & ~cb).sum()), int((~ca & cb).sum())
    return n10, n01, (binomtest(n10, n10 + n01, 0.5).pvalue if n10 + n01 else 1.0)

PAIRS = [("lp_medsiglip", "lp_imagenet", "H1 linear probe"), ("mlp_medsiglip", "mlp_imagenet", "H1 MLP head"),
         ("imagenet+text", "mlp_imagenet", "text gain (frozen ImageNet)"),
         ("imagenet+medsiglip", "mlp_imagenet", "MedSigLIP-embedding gain"),
         ("imagenet+medsiglip", "imagenet+text", "H2: MedSigLIP embedding vs text"),
         ("imagenet+medsiglip", "imagenet+medsiglip_shuffled", "control: image-specific?"),
         ("imagenet+medsiglip+text", "imagenet+medsiglip", "text beyond MedSigLIP"),
         ("medsiglip+text", "mlp_medsiglip", "text on the medical backbone")]

for name in RUN_ORDER:
    tag = TEXT[name]["tag"]; d = D[name]
    r = res[(res["dataset"] == name) & res["text_tag"].isin(["", tag])]
    tab = r.groupby(["variant", "frac"])["acc"].agg(["mean", "std", "count"])
    tab = tab.apply(lambda x: f"{x['mean']:.1f} ± {0 if np.isnan(x['std']) else x['std']:.1f} (n={int(x['count'])})", axis=1).unstack("frac")
    tab = tab.reindex([v for v in order if v in set(r["variant"])]); tab.columns = [f"{c*100:g}%" for c in tab.columns]
    pv2 = f"{d['cfg']['pipeline_dir']}/results_v2.csv"
    pr = pd.read_csv(pv2 if os.path.exists(pv2) else f"{d['cfg']['pipeline_dir']}/results_fast.csv")
    ref = pr[pr["variant"] == "linear_probe"].groupby("frac")["acc"].mean()
    print(f"\n=== {name}: accuracy % (test n={len(d['test'])}, majority {100 * np.bincount(d['y'][d['test']]).max() / len(d['test']):.1f}%) ===")
    print(tab.to_string()); print("pipeline raw-feature linear_probe:", {f"{k*100:g}%": round(v, 1) for k, v in ref.items()})
    tab.to_csv(f"{OUT_DIR}/table_acc_{name}.csv")
    print("\npaired comparisons (mean diff over seeds; t-test over seeds; McNemar pooled over seeds):")
    for f in d["cfg"]["fractions"]:
        pv = r[r["frac"].round(4) == round(f, 4)].pivot_table(index="seed", columns="variant", values="acc")
        for a, b, label in PAIRS:
            if a in pv and b in pv:
                dd = (pv[a] - pv[b]).dropna()
                tp = stats.ttest_rel(pv.loc[dd.index, a], pv.loc[dd.index, b]).pvalue if len(dd) > 1 else float("nan")
                mc = mcnemar_pooled(name, f, a, b)
                mtxt = f"McNemar {mc[0]}:{mc[1]} p={mc[2]:.3g}" if mc else ""
                print(f"  {f*100:>5g}%  {label:<32} {a} - {b}: {dd.mean():+5.1f} pts ({(dd > 0).sum()}/{len(dd)}), t p={tp:.3f}  {mtxt}")


## Next
- If `lp_medsiglip` is far above `lp_imagenet` (H1), run DKA on the MedSigLIP vision tower next (its 448 px input gives a 32x32 patch grid, which also makes the large-vs-small kernel question (H4) meaningful).
- Feed `preds.csv` from here, `mm_dka/dka_preds_v2.csv` and the pipelines' `preds_v2.csv` into `analysis_h8.ipynb` for the complementarity analysis.